# MIMIC Target Dataset EDA

Exploratory analysis for all available MIMIC target datasets under:

- `obs24_target8_gap0`
- `obs24_target8_gap2`

The notebook expects each target folder to contain `labels.csv`, `records.csv`, and `dataset_metadata.json`.

In [ ]:
from pathlib import Path
import json
import os
import warnings

os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
try:
    import seaborn as sns
except ModuleNotFoundError:
    sns = None

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 120)
pd.set_option("display.max_colwidth", 140)
plt.style.use("ggplot")

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data" / "mimic_targets").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_ROOT = PROJECT_ROOT / "data" / "mimic_targets"
SETTINGS = ["obs24_target8_gap0", "obs24_target8_gap2"]

# Set to an integer such as 1_000_000 if the full records files are too large for local memory.
RECORD_NROWS = None

DATA_ROOT

## Load Datasets

In [ ]:
def discover_target_dirs(data_root=DATA_ROOT, settings=SETTINGS):
    rows = []
    for setting in settings:
        setting_dir = data_root / setting
        if not setting_dir.exists():
            continue
        for target_dir in sorted(p for p in setting_dir.iterdir() if p.is_dir()):
            rows.append({"setting": setting, "target": target_dir.name, "path": target_dir})
    return pd.DataFrame(rows)


target_dirs = discover_target_dirs()
target_dirs

In [ ]:
def load_one_dataset(setting, target, path, record_nrows=RECORD_NROWS):
    labels_path = path / "labels.csv"
    records_path = path / "records.csv"
    metadata_path = path / "dataset_metadata.json"

    labels = pd.read_csv(labels_path)
    records = pd.read_csv(records_path, nrows=record_nrows)
    metadata = json.loads(metadata_path.read_text()) if metadata_path.exists() else {}

    labels["setting"] = setting
    labels["target"] = target
    labels["label"] = labels["label"].astype(bool)

    records["setting"] = setting
    records["target"] = target
    records["value"] = pd.to_numeric(records["value"], errors="coerce")
    records["timestamp"] = pd.to_datetime(records["timestamp"], errors="coerce")

    return labels, records, metadata


labels_by_dataset = {}
records_by_dataset = {}
metadata_by_dataset = {}

for row in target_dirs.itertuples(index=False):
    key = (row.setting, row.target)
    labels_by_dataset[key], records_by_dataset[key], metadata_by_dataset[key] = load_one_dataset(
        row.setting, row.target, row.path
    )

labels = pd.concat(labels_by_dataset.values(), ignore_index=True)
records = pd.concat(records_by_dataset.values(), ignore_index=True)

labels.shape, records.shape

## Metadata And Cohort Overview

In [ ]:
def metadata_summary(metadata_by_dataset):
    rows = []
    for (setting, target), metadata in metadata_by_dataset.items():
        label_counts = metadata.get("label_counts", {}) or {}
        window = metadata.get("window", {}) or {}
        rows.append(
            {
                "setting": setting,
                "target": target,
                "observation_hours": window.get("observation_hours"),
                "prediction_hours": window.get("prediction_hours"),
                "gap_hours": window.get("gap_hours"),
                "metadata_candidate_icus": metadata.get("n_candidate_icus"),
                "metadata_labeled_icus": metadata.get("n_labeled_icus"),
                "metadata_records": metadata.get("n_records"),
                "metadata_positive": label_counts.get("true", label_counts.get(True)),
                "metadata_negative": label_counts.get("false", label_counts.get(False)),
                "n_variables_metadata": len(metadata.get("record_counts_by_variable", {}) or {}),
                "target_definition": (metadata.get("target_metadata", {}) or {}).get("target_definition"),
            }
        )
    out = pd.DataFrame(rows)
    out["metadata_prevalence"] = out["metadata_positive"] / out["metadata_labeled_icus"]
    return out.sort_values(["target", "setting"])


meta_df = metadata_summary(metadata_by_dataset)
meta_df

In [ ]:
loaded_summary = (
    labels.groupby(["setting", "target"])
    .agg(
        loaded_patients=("patient_id", "nunique"),
        loaded_positive=("label", "sum"),
        loaded_negative=("label", lambda s: (~s).sum()),
    )
    .reset_index()
)
record_summary = (
    records.groupby(["setting", "target"])
    .agg(
        loaded_records=("patient_id", "size"),
        patients_with_records=("patient_id", "nunique"),
        loaded_variables=("variable", "nunique"),
        first_timestamp=("timestamp", "min"),
        last_timestamp=("timestamp", "max"),
    )
    .reset_index()
)

cohort_df = (
    meta_df.merge(loaded_summary, on=["setting", "target"], how="left")
    .merge(record_summary, on=["setting", "target"], how="left")
)
cohort_df["loaded_prevalence"] = cohort_df["loaded_positive"] / cohort_df["loaded_patients"]
cohort_df["metadata_vs_loaded_record_ratio"] = cohort_df["loaded_records"] / cohort_df["metadata_records"]
cohort_df

## Label Balance

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5), constrained_layout=True)

cohort_df.pivot(index="target", columns="setting", values="loaded_patients").plot(kind="bar", ax=axes[0])
axes[0].set_title("Loaded labeled patients")
axes[0].set_xlabel("")
axes[0].set_ylabel("Patients")
axes[0].tick_params(axis="x", rotation=20)

cohort_df.pivot(index="target", columns="setting", values="loaded_prevalence").plot(kind="bar", ax=axes[1])
axes[1].set_title("Positive label prevalence")
axes[1].set_xlabel("")
axes[1].set_ylabel("Prevalence")
axes[1].tick_params(axis="x", rotation=20)
axes[1].yaxis.set_major_formatter(lambda x, _: f"{x:.0%}")

plt.show()

In [ ]:
label_counts = (
    labels.assign(label_name=lambda d: np.where(d["label"], "positive", "negative"))
    .groupby(["setting", "target", "label_name"])
    .size()
    .rename("count")
    .reset_index()
)

targets = sorted(label_counts["target"].unique())
fig, axes = plt.subplots(1, len(targets), figsize=(5 * len(targets), 4), constrained_layout=True)
axes = np.atleast_1d(axes)
for ax, target in zip(axes, targets):
    plot_df = label_counts[label_counts["target"] == target]
    plot_df.pivot(index="label_name", columns="setting", values="count").plot(kind="bar", ax=ax)
    ax.set_title(target)
    ax.set_xlabel("")
    ax.set_ylabel("Patients")
    ax.tick_params(axis="x", rotation=0)
plt.show()

label_counts.pivot_table(index=["target", "setting"], columns="label_name", values="count", fill_value=0)

## Record And Variable Coverage

In [ ]:
records_with_labels = records.merge(labels, on=["setting", "target", "patient_id"], how="left")
records_with_labels["label_name"] = np.where(records_with_labels["label"], "positive", "negative")

variable_counts = (
    records_with_labels.groupby(["setting", "target", "variable"])
    .agg(
        n_records=("value", "size"),
        n_patients=("patient_id", "nunique"),
        missing_values=("value", lambda s: s.isna().sum()),
        median_value=("value", "median"),
        q1_value=("value", lambda s: s.quantile(0.25)),
        q3_value=("value", lambda s: s.quantile(0.75)),
    )
    .reset_index()
)
variable_counts["missing_rate"] = variable_counts["missing_values"] / variable_counts["n_records"]
variable_counts.sort_values(["target", "setting", "n_records"], ascending=[True, True, False]).head(30)

### Sampling Frequency For One Target And Setting

In [ ]:
# Choose any available dataset pair, for example:
# SELECTED_SETTING = "obs24_target8_gap0"
# SELECTED_TARGET = "hypotension"
SELECTED_SETTING = "obs24_target8_gap0"
SELECTED_TARGET = "hypotension"

selected_key = (SELECTED_SETTING, SELECTED_TARGET)
if selected_key not in records_by_dataset:
    selected_key = next(iter(records_by_dataset))
    SELECTED_SETTING, SELECTED_TARGET = selected_key
    print(f"Selected pair was not found. Using {SELECTED_SETTING}/{SELECTED_TARGET} instead.")

selected_records = records_by_dataset[selected_key].copy()
selected_labels = labels_by_dataset[selected_key].copy()
selected_metadata = metadata_by_dataset.get(selected_key, {})
observation_hours = (selected_metadata.get("window", {}) or {}).get("observation_hours")

if observation_hours is None:
    observation_hours = (
        selected_records["timestamp"].max() - selected_records["timestamp"].min()
    ).total_seconds() / 3600

n_labeled_stays = selected_labels["patient_id"].nunique()

samples_per_observed_stay = (
    selected_records.groupby(["variable", "patient_id"])
    .agg(
        n_samples=("value", "size"),
        first_timestamp=("timestamp", "min"),
        last_timestamp=("timestamp", "max"),
    )
    .reset_index()
)
samples_per_observed_stay["observed_span_hours"] = (
    samples_per_observed_stay["last_timestamp"] - samples_per_observed_stay["first_timestamp"]
).dt.total_seconds() / 3600
samples_per_observed_stay["hours_between_measurements"] = np.where(
    samples_per_observed_stay["n_samples"] > 1,
    samples_per_observed_stay["observed_span_hours"] / (samples_per_observed_stay["n_samples"] - 1),
    np.nan,
)
samples_per_observed_stay["measurements_per_hour"] = samples_per_observed_stay["n_samples"] / observation_hours

variable_sampling_summary = (
    samples_per_observed_stay.groupby("variable")
    .agg(
        stays_with_variable=("patient_id", "nunique"),
        total_samples=("n_samples", "sum"),
        mean_samples_per_observed_stay=("n_samples", "mean"),
        median_samples_per_observed_stay=("n_samples", "median"),
        p90_samples_per_observed_stay=("n_samples", lambda s: s.quantile(0.9)),
        mean_measurements_per_hour_when_observed=("measurements_per_hour", "mean"),
        mean_hours_between_measurements=("hours_between_measurements", "mean"),
    )
    .reset_index()
)
variable_sampling_summary["total_labeled_stays"] = n_labeled_stays
variable_sampling_summary["stay_coverage"] = variable_sampling_summary["stays_with_variable"] / n_labeled_stays
variable_sampling_summary["mean_samples_per_labeled_stay"] = (
    variable_sampling_summary["total_samples"] / n_labeled_stays
)
variable_sampling_summary["mean_measurements_per_hour_per_labeled_stay"] = (
    variable_sampling_summary["mean_samples_per_labeled_stay"] / observation_hours
)

ordered_columns = [
    "variable",
    "total_labeled_stays",
    "stays_with_variable",
    "stay_coverage",
    "total_samples",
    "mean_samples_per_labeled_stay",
    "mean_samples_per_observed_stay",
    "median_samples_per_observed_stay",
    "p90_samples_per_observed_stay",
    "mean_measurements_per_hour_per_labeled_stay",
    "mean_measurements_per_hour_when_observed",
    "mean_hours_between_measurements",
]

print(
    f"Sampling summary for {SELECTED_SETTING}/{SELECTED_TARGET} "
    f"over a {observation_hours:g}-hour observation window"
)
variable_sampling_summary[ordered_columns].sort_values(
    "mean_samples_per_labeled_stay", ascending=False
).reset_index(drop=True)


In [ ]:
for target in sorted(records["target"].unique()):
    plot_df = variable_counts[variable_counts["target"] == target].sort_values("n_records", ascending=False)
    order = plot_df.groupby("variable")["n_records"].sum().sort_values(ascending=False).index
    ax = (
        plot_df.pivot(index="variable", columns="setting", values="n_records")
        .reindex(order)
        .plot(kind="barh", figsize=(12, max(4, 0.28 * len(order))))
    )
    ax.invert_yaxis()
    ax.set_title(f"Record counts by variable: {target}")
    ax.set_xlabel("Records")
    ax.set_ylabel("")
    plt.tight_layout()
    plt.show()

In [ ]:
patient_coverage = (
    records_with_labels.groupby(["setting", "target", "label_name", "patient_id"])
    .agg(
        n_records=("value", "size"),
        n_variables=("variable", "nunique"),
    )
    .reset_index()
)

coverage_summary = (
    patient_coverage.groupby(["setting", "target", "label_name"])
    .agg(
        patients=("patient_id", "nunique"),
        median_records_per_patient=("n_records", "median"),
        p90_records_per_patient=("n_records", lambda s: s.quantile(0.9)),
        median_variables_per_patient=("n_variables", "median"),
        p90_variables_per_patient=("n_variables", lambda s: s.quantile(0.9)),
    )
    .reset_index()
)
coverage_summary

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5), constrained_layout=True)
patient_coverage.assign(group=lambda d: d["target"] + "\n" + d["setting"]).boxplot(
    column="n_records", by="group", showfliers=False, ax=axes[0], grid=False
)
axes[0].set_title("Records per patient")
axes[0].set_xlabel("")
axes[0].set_ylabel("Records")
axes[0].tick_params(axis="x", rotation=20)

patient_coverage.assign(group=lambda d: d["target"] + "\n" + d["setting"]).boxplot(
    column="n_variables", by="group", showfliers=False, ax=axes[1], grid=False
)
axes[1].set_title("Variables per patient")
axes[1].set_xlabel("")
axes[1].set_ylabel("Variables")
axes[1].tick_params(axis="x", rotation=20)

fig.suptitle("")
plt.show()

## Time Coverage

In [ ]:
records_with_labels["hours_since_anchor"] = (
    records_with_labels["timestamp"] - pd.Timestamp("2000-01-01 00:00:00")
).dt.total_seconds() / 3600

time_summary = (
    records_with_labels.groupby(["setting", "target"])
    .agg(
        min_hour=("hours_since_anchor", "min"),
        q1_hour=("hours_since_anchor", lambda s: s.quantile(0.25)),
        median_hour=("hours_since_anchor", "median"),
        q3_hour=("hours_since_anchor", lambda s: s.quantile(0.75)),
        max_hour=("hours_since_anchor", "max"),
    )
    .reset_index()
)
time_summary

In [ ]:
targets = sorted(records_with_labels["target"].unique())
fig, axes = plt.subplots(1, len(targets), figsize=(5 * len(targets), 4), constrained_layout=True)
axes = np.atleast_1d(axes)
for ax, target in zip(axes, targets):
    for setting, group in records_with_labels[records_with_labels["target"] == target].groupby("setting"):
        ax.hist(group["hours_since_anchor"].dropna(), bins=48, alpha=0.55, label=setting)
    ax.axvline(24, color="black", linestyle="--", linewidth=1)
    ax.set_title(target)
    ax.set_xlabel("Hours since ICU admission anchor")
    ax.set_ylabel("Records")
    ax.legend()
plt.show()

## Value Distributions

In [ ]:
value_summary = (
    records_with_labels.groupby(["setting", "target", "variable"])["value"]
    .describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99])
    .reset_index()
)
value_summary.sort_values(["target", "setting", "count"], ascending=[True, True, False]).head(40)

In [ ]:
TOP_N_VARIABLES = 12

for target in sorted(records_with_labels["target"].unique()):
    top_vars = (
        records_with_labels.loc[records_with_labels["target"] == target, "variable"]
        .value_counts()
        .head(TOP_N_VARIABLES)
        .index
    )
    plot_df = records_with_labels[
        (records_with_labels["target"] == target) & records_with_labels["variable"].isin(top_vars)
    ].copy()
    plot_df["variable"] = pd.Categorical(plot_df["variable"], categories=top_vars, ordered=True)

    summary_plot = (
        plot_df.groupby(["variable", "setting", "label_name"])["value"]
        .median()
        .rename("median_value")
        .reset_index()
    )
    for label_name in sorted(summary_plot["label_name"].dropna().unique()):
        ax = (
            summary_plot[summary_plot["label_name"] == label_name]
            .pivot(index="variable", columns="setting", values="median_value")
            .reindex(top_vars)
            .plot(kind="barh", figsize=(12, max(4, 0.35 * len(top_vars))))
        )
        ax.invert_yaxis()
        ax.set_title(f"Median values for top variables: {target} ({label_name})")
        ax.set_xlabel("Median value")
        ax.set_ylabel("")
        plt.tight_layout()
        plt.show()

## Gap0 vs Gap2 Comparison

In [ ]:
comparison_metrics = [
    "loaded_patients",
    "loaded_positive",
    "loaded_prevalence",
    "loaded_records",
    "patients_with_records",
    "loaded_variables",
]

gap_comparison = cohort_df.pivot(index="target", columns="setting", values=comparison_metrics)
gap_comparison

In [ ]:
delta_rows = []
for target, group in cohort_df.groupby("target"):
    by_setting = group.set_index("setting")
    if set(SETTINGS).issubset(by_setting.index):
        row = {"target": target}
        for metric in comparison_metrics:
            gap0 = by_setting.loc["obs24_target8_gap0", metric]
            gap2 = by_setting.loc["obs24_target8_gap2", metric]
            row[f"{metric}_gap2_minus_gap0"] = gap2 - gap0
            row[f"{metric}_gap2_over_gap0"] = gap2 / gap0 if gap0 else np.nan
        delta_rows.append(row)

gap_delta = pd.DataFrame(delta_rows)
gap_delta

## Data Quality Checks

In [ ]:
quality_rows = []
for (setting, target), dataset_labels in labels_by_dataset.items():
    dataset_records = records_by_dataset[(setting, target)]
    label_patients = set(dataset_labels["patient_id"])
    record_patients = set(dataset_records["patient_id"])
    duplicated_labels = dataset_labels.duplicated("patient_id").sum()
    quality_rows.append(
        {
            "setting": setting,
            "target": target,
            "duplicated_label_patient_rows": duplicated_labels,
            "records_with_missing_patient_id": dataset_records["patient_id"].isna().sum(),
            "records_with_missing_variable": dataset_records["variable"].isna().sum(),
            "records_with_missing_timestamp": dataset_records["timestamp"].isna().sum(),
            "records_with_missing_value": dataset_records["value"].isna().sum(),
            "patients_in_labels_not_records": len(label_patients - record_patients),
            "patients_in_records_not_labels": len(record_patients - label_patients),
        }
    )

quality_df = pd.DataFrame(quality_rows).sort_values(["target", "setting"])
quality_df

## Export Summary Tables

In [ ]:
OUTPUT_DIR = PROJECT_ROOT / "notebooks" / "eda_outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

cohort_df.to_csv(OUTPUT_DIR / "mimic_target_cohort_summary.csv", index=False)
variable_counts.to_csv(OUTPUT_DIR / "mimic_target_variable_summary.csv", index=False)
coverage_summary.to_csv(OUTPUT_DIR / "mimic_target_patient_coverage_summary.csv", index=False)
time_summary.to_csv(OUTPUT_DIR / "mimic_target_time_summary.csv", index=False)
value_summary.to_csv(OUTPUT_DIR / "mimic_target_value_summary.csv", index=False)
quality_df.to_csv(OUTPUT_DIR / "mimic_target_quality_checks.csv", index=False)
gap_delta.to_csv(OUTPUT_DIR / "mimic_target_gap_delta_summary.csv", index=False)

sorted(p.name for p in OUTPUT_DIR.glob("*.csv"))